# TP2 : descente de gradient, critères d'arrêt et choix du pas
**Cours :** Algorithmes d'optimisation, séance 6 · **Intervenant :** Serigne Faye

## Objectifs
- coder vous-même la descente de gradient à pas fixe ;
- **prévoir** le comportement de l'algorithme selon le pas, puis le vérifier ;
- coder **plusieurs critères d'arrêt** et comprendre ce que chacun garantit réellement (et ses pièges) ;
- coder une recherche linéaire (backtracking) et le pas optimal ;
- choisir et justifier une configuration pour atteindre une précision donnée.

## Consignes
Exécutez les cellules dans l'ordre. Complétez les cellules marquées **À COMPLÉTER**. Les cellules de test vous disent immédiatement si votre code est correct. Répondez aux questions dans les cellules de texte : elles comptent autant que le code.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## Fonction de travail
Sauf mention contraire, on minimise $f(x, y) = x^2 + 10y^2$, de gradient $(2x, 20y)^\top$ et de minimum $x^* = (0, 0)$, en partant de $x_0 = (1, 1)$.

In [ ]:
def f(p): return p[0]**2 + 10*p[1]**2
def grad(p): return np.array([2*p[0], 20*p[1]])
x0 = np.array([1.0, 1.0])
x_star = np.zeros(2)

## Partie 1 : la descente à pas fixe, écrite par vous
**Exercice 1.1.** Compléter la fonction `descente_pas_fixe`. Elle renvoie le dernier point, la liste des itérés et la liste des valeurs de $f$.

In [ ]:
def descente_pas_fixe(f, grad, x0, alpha, max_iter):
    x = np.array(x0, dtype=float)
    xs, fs = [x.copy()], [f(x)]
    for k in range(max_iter):
        # À COMPLÉTER : mise à jour x <- x - alpha * grad(x)
        pass
        xs.append(x.copy()); fs.append(f(x))
    return x, np.array(xs), np.array(fs)

In [ ]:
x_fin, xs, fs = descente_pas_fixe(f, grad, x0, 0.05, 200)
print(x_fin)
assert np.linalg.norm(x_fin - x_star) < 1e-6, "La descente ne converge pas : vérifiez la mise à jour"
assert np.all(np.diff(fs) <= 1e-12), "f doit décroître à chaque itération pour ce pas"

# vérification avec l'exemple de la séance 5 : f(x) = (x-3)^2, x0 = 0, alpha = 0.1, 20 itérations
x1, _, _ = descente_pas_fixe(lambda p: (p[0] - 3)**2, lambda p: np.array([2*(p[0] - 3)]), [0.0], 0.1, 20)
print(x1)   # attendu : environ 2.9654
assert abs(x1[0] - 2.9654) < 1e-3
print("OK")

## Partie 2 : prévoir, puis vérifier
Pour $f(x, y) = x^2 + 10y^2$, la plus grande valeur propre de la hessienne $\mathrm{diag}(2, 20)$ est $\lambda_{\max} = 20$. D'après le cours, la descente à pas constant converge si $0 < \alpha < 2/\lambda_{\max}$.

**Exercice 2.1.** **Avant d'exécuter quoi que ce soit**, écrivez dans le dictionnaire `predictions` le comportement que vous attendez pour chaque pas (300 itérations) : `"converge"`, `"lent"`, `"oscille"` ou `"diverge"`.

In [ ]:
pas_liste = [0.001, 0.01, 0.05, 0.09, 0.1, 0.11]
# À COMPLÉTER : une chaîne par pas, par exemple "lent"
predictions = {0.001: "", 0.01: "", 0.05: "", 0.09: "", 0.1: "", 0.11: ""}

La fonction `comportement` (fournie) lance la descente et classe le résultat. Comparez avec vos prévisions.

In [ ]:
def comportement(alpha, n=300):
    _, xs, _ = descente_pas_fixe(f, grad, x0, alpha, n)
    d = np.linalg.norm(xs - x_star, axis=1)          # distance au minimum
    if not np.isfinite(d[-1]) or d[-1] > 1e6: return "diverge"
    if d[-1] < 1e-6: return "converge"
    if d[-1] < 0.99 * d[-50]: return "lent"           # la distance diminue, mais lentement
    return "oscille"

print("   pas | prévu    | observé  | distance finale")
for a in pas_liste:
    obs = comportement(a)
    d = np.linalg.norm(descente_pas_fixe(f, grad, x0, a, 300)[0] - x_star)
    print(f"{a:6.3f} | {predictions[a]:8s} | {obs:8s} | {d:10.2e}   {'OK' if predictions[a] == obs else '<-- écart'}")

**Question 2.2.** Pour chaque écart entre votre prévision et le résultat, expliquez ce que vous aviez mal anticipé. Que se passe-t-il exactement pour $\alpha = 0{,}1$ ? (Indice : calculez le facteur par lequel la composante $y$ est multipliée à chaque itération, $1 - 20\alpha$.)

**Réponse :** *(à compléter)*

## Partie 3 : les critères d'arrêt
Jusqu'ici, on a lancé un nombre fixe d'itérations. En pratique, il faut décider **quand s'arrêter**. Quatre critères, avec un seuil `tol` :

| nom | l'algorithme s'arrête quand |
|---|---|
| `"grad"` | $\|\nabla f(x_{k+1})\| < \text{tol}$ |
| `"pas"` | $\|x_{k+1} - x_k\| < \text{tol}$ |
| `"fval"` | $\|f(x_{k+1}) - f(x_k)\| < \text{tol}$ |
| `"grad_rel"` | $\|\nabla f(x_{k+1})\| < \text{tol} \times \|\nabla f(x_0)\|$ (critère **relatif**) |

En plus, on garde toujours un nombre maximal d'itérations `max_iter` et une détection de divergence.

**Exercice 3.1.** Compléter `test_arret`, qui renvoie `True` quand l'algorithme doit s'arrêter.

In [ ]:
def test_arret(critere, tol, x_old, x_new, f_old, f_new, g_new, g0_norm):
    if critere == "grad":
        return False   # À COMPLÉTER : norme de g_new < tol
    if critere == "pas":
        return False   # À COMPLÉTER : norme de (x_new - x_old) < tol
    if critere == "fval":
        return False   # À COMPLÉTER : |f_new - f_old| < tol
    if critere == "grad_rel":
        return False   # À COMPLÉTER : norme de g_new < tol * g0_norm
    return False

In [ ]:
g_petit, g_grand, z = np.array([1e-7, 0.0]), np.array([1.0, 0.0]), np.zeros(2)
assert test_arret("grad", 1e-6, z, z, 0, 0, g_petit, 1.0)
assert not test_arret("grad", 1e-6, z, z, 0, 0, g_grand, 1.0)
assert test_arret("pas", 1e-6, z, z + 1e-8, 0, 0, g_grand, 1.0)
assert not test_arret("pas", 1e-6, z, z + 1.0, 0, 0, g_grand, 1.0)
assert test_arret("fval", 1e-6, z, z, 1.0, 1.0 - 1e-9, g_grand, 1.0)
assert not test_arret("fval", 1e-6, z, z, 1.0, 0.5, g_grand, 1.0)
assert test_arret("grad_rel", 1e-6, z, z, 0, 0, g_petit, 1.0)
assert not test_arret("grad_rel", 1e-6, z, z, 0, 0, g_petit, 1e-3)
print("test_arret : OK")

La fonction `descente` (fournie) utilise votre `test_arret`. Le paramètre `pas` peut être un nombre ou une règle `regle(x, g)` qui renvoie le pas à chaque itération (on s'en servira en partie 4). Elle renvoie le dernier point et un dictionnaire avec le nombre d'itérations, le statut (`converge`, `max_iter` ou `divergence`) et l'historique.

In [ ]:
def descente(f, grad, x0, pas, max_iter=10000, tol=1e-6, critere="grad"):
    x = np.array(x0, dtype=float)
    g = grad(x); g0_norm = np.linalg.norm(g)
    xs, fs = [x.copy()], [f(x)]
    statut = "max_iter"
    if critere == "grad" and test_arret("grad", tol, x, x, f(x), f(x), g, g0_norm):
        return x, {"xs": np.array(xs), "fs": np.array(fs), "iters": 0, "statut": "converge"}
    for k in range(max_iter):
        t = pas(x, g) if callable(pas) else pas
        x_new = x - t * g
        if not np.all(np.isfinite(x_new)) or np.linalg.norm(x_new) > 1e8:
            statut = "divergence"; break
        g_new = grad(x_new)
        x_old, f_old, f_new = x, f(x), f(x_new)
        x, g = x_new, g_new
        xs.append(x.copy()); fs.append(f_new)
        if test_arret(critere, tol, x_old, x, f_old, f_new, g, g0_norm):
            statut = "converge"; break
    return x, {"xs": np.array(xs), "fs": np.array(fs), "iters": len(xs) - 1, "statut": statut}

### 3.2 Les quatre critères sur le même problème
**Exercice 3.2.** Avec $\alpha = 0{,}05$ et `tol = 1e-6`, lancer `descente` pour chacun des quatre critères et afficher le nombre d'itérations, le statut et la **précision réellement atteinte** $\|x - x^*\|$.

In [ ]:
print("critère  | itérations | statut   | précision réelle ‖x − x*‖")
for critere in ["grad", "pas", "fval", "grad_rel"]:
    # À COMPLÉTER : lancer descente(f, grad, x0, 0.05, tol=1e-6, critere=critere) et afficher une ligne
    pass

**Question 3.3.** Le seuil `tol` est le même pour les quatre critères, pourtant la précision obtenue est très différente. Pourquoi ? Quel critère s'arrête le plus tôt, et que garantit-il réellement sur la distance au minimum ?

**Réponse :** *(à compléter)*

### 3.4 Piège n°1 : un pas minuscule
**Exercice 3.4.** Lancer `descente` avec $\alpha = 10^{-9}$ (`max_iter=1000`, `tol=1e-6`) pour les critères `"grad"`, `"pas"` et `"fval"`. Afficher le statut, le nombre d'itérations et la distance au minimum.

In [ ]:
for critere in ["grad", "pas", "fval"]:
    # À COMPLÉTER
    pass

**Question 3.5.** Que déclarent `pas` et `fval` ? Est-ce vrai ? Que fait `grad` ? Quel est le danger pour quelqu'un qui n'afficherait que « l'algorithme a convergé » ?

**Réponse :** *(à compléter)*

### 3.6 Piège n°2 : une fonction à une autre échelle
On considère la même fonction multipliée par $10^{-8}$, avec le pas multiplié par $10^{8}$ (la dynamique de l'algorithme est identique).

In [ ]:
f_s = lambda p: 1e-8 * f(p)
grad_s = lambda p: 1e-8 * grad(p)
alpha_s = 0.05 / 1e-8

**Exercice 3.7.** Lancer `descente` sur `f_s` avec `tol = 1e-6` pour les critères `"grad"` et `"grad_rel"`. Comparer le nombre d'itérations et la précision atteinte à ce que vous avez obtenu en 3.2.

In [ ]:
for critere in ["grad", "grad_rel"]:
    # À COMPLÉTER
    pass

**Question 3.8.** Que se passe-t-il avec le critère absolu `grad` ? Pourquoi le critère relatif `grad_rel` est-il plus robuste ?

**Réponse :** *(à compléter)*

### 3.9 Divergence et nombre maximal d'itérations
**Exercice 3.9.** Lancer `descente` avec $\alpha = 0{,}11$ et le critère `"grad"`. Quel statut obtient-on, et après combien d'itérations ?

In [ ]:
# À COMPLÉTER

**Question 3.10.** Pourquoi faut-il toujours prévoir un `max_iter` et une détection de divergence, quel que soit le critère d'arrêt choisi ?

**Réponse :** *(à compléter)*

## Partie 4 : recherche linéaire (backtracking)
Plutôt que de fixer le pas, on le choisit à chaque itération : on part de $t = 1$ et on le divise par 2 tant que la condition d'Armijo n'est pas satisfaite :
$$f(x - t g) \leq f(x) - c\, t\, \|g\|^2, \qquad g = \nabla f(x), \quad c = 10^{-4}.$$

**Exercice 4.1.** Écrire `backtracking(f, grad, x)` qui renvoie le pas $t$.

In [ ]:
def backtracking(f, grad, x, t0=1.0, beta=0.5, c=1e-4):
    g = grad(x)
    t = t0
    # À COMPLÉTER : boucle while qui divise t par beta tant que la condition d'Armijo n'est pas vérifiée
    return t

In [ ]:
t = backtracking(f, grad, x0)
g = grad(x0)
assert f(x0 - t * g) <= f(x0), "Le pas trouvé doit faire diminuer f"
assert 0 < t <= 1
print("pas initial choisi :", t)

**Exercice 4.2.** Comparer, avec le critère `"grad_rel"` (`tol = 1e-6`) : le pas fixe $0{,}05$, le pas fixe $0{,}09$ et le backtracking (`pas=lambda x, g: backtracking(f, grad, x)`). Afficher le nombre d'itérations et la précision.

In [ ]:
configs = {"fixe 0.05": 0.05, "fixe 0.09": 0.09, "backtracking": lambda x, g: backtracking(f, grad, x)}
for nom, pas in configs.items():
    # À COMPLÉTER : lancer descente(...) et afficher une ligne
    pass

**Question 4.3.** Que constatez-vous ? Quel est l'avantage du backtracking, et quel est son coût caché que le nombre d'itérations ne montre pas ?

**Réponse :** *(à compléter)*

## Partie 5 : pas optimal pour une fonction quadratique
Pour $f(x) = \tfrac12 x^\top A x$ avec $A = \mathrm{diag}(2, 20)$ (c'est notre fonction), la recherche linéaire exacte dans la direction $-g$ donne
$$t_k = \frac{g^\top g}{g^\top A g}.$$

**Exercice 5.1.** Écrire la règle `regle_optimale(x, g)`, puis comparer pas fixe $0{,}05$, pas optimal et backtracking depuis $x_0' = (1\,;\,0{,}1)$ avec le critère `"grad_rel"`. Tracer les trajectoires du pas optimal et du pas fixe sur les courbes de niveau.

In [ ]:
A = np.diag([2.0, 20.0])
def regle_optimale(x, g):
    # À COMPLÉTER : renvoyer g^T g / g^T A g
    return 0.05

xb = np.array([1.0, 0.1])
res = {}
for nom, pas in {"fixe 0.05": 0.05, "optimal": regle_optimale, "backtracking": lambda x, g: backtracking(f, grad, x)}.items():
    x, res[nom] = descente(f, grad, xb, pas, tol=1e-6, critere="grad_rel")
    print(f"{nom:13s} : {res[nom]['iters']:4d} itérations")

xs = np.linspace(-1.2, 1.2, 200); X, Y = np.meshgrid(xs, xs)
plt.figure(figsize=(6, 6)); plt.contour(X, Y, X**2 + 10*Y**2, levels=20, alpha=0.4)
# À COMPLÉTER : tracer res["optimal"]["xs"] et res["fixe 0.05"]["xs"] avec plt.plot(..., "o-")
plt.legend(); plt.show()

**Question 5.2.** Décrivez les trajectoires. Pourquoi le pas optimal « zigzague »-t-il dans la vallée ?

**Réponse :** *(à compléter)*

## Partie 6 : le défi
On minimise $f_\kappa(x, y) = \tfrac12 (x^2 + 50 y^2)$ (conditionnement $\kappa = 50$) depuis $(1\,;\,0{,}02)$. **Objectif :** atteindre une distance au minimum **inférieure à $10^{-4}$** avec le moins d'itérations possible, en choisissant la règle de pas et le seuil `tol` (critère `"grad_rel"`).

La fonction `evalue` (fournie) renvoie le nombre d'itérations, la distance au minimum et le statut.

In [ ]:
kappa = 50
A_k = np.diag([1.0, kappa])
f_k = lambda p: 0.5 * p @ A_k @ p
grad_k = lambda p: A_k @ p
x_k0 = np.array([1.0, 1.0 / kappa])

def evalue(pas, tol):
    x, h = descente(f_k, grad_k, x_k0, pas, max_iter=20000, tol=tol, critere="grad_rel")
    return h["iters"], np.linalg.norm(x), h["statut"]

print(evalue(0.03, 1e-4))   # exemple d'appel

**Exercice 6.1.** Pour chaque configuration et chaque `tol` de la liste, relever le nombre d'itérations et la distance obtenue, puis repérer pour chaque configuration la **plus grande** valeur de `tol` qui atteint la précision demandée.

In [ ]:
configs = {
    "fixe 2/(1+kappa)": 2 / (1 + kappa),
    "fixe 0.03": 0.03,
    "backtracking": lambda x, g: backtracking(f_k, grad_k, x),
    "optimal": lambda x, g: (g @ g) / (g @ A_k @ g),
}
for nom, pas in configs.items():
    for tol in [1e-2, 1e-3, 1e-4, 1e-5]:
        # À COMPLÉTER : appeler evalue(pas, tol) et afficher nom, tol, itérations, distance, et si la distance est < 1e-4
        pass

**Question 6.2.** Quelle configuration (règle de pas et `tol`) choisissez-vous, et pourquoi ? Que remarquez-vous sur « fixe 2/(1+kappa) » et « optimal » ? Quel critère de choix avez-vous utilisé en plus du nombre d'itérations ?

**Réponse :** *(à compléter)*

## Partie 7 (bonus) : pour aller plus loin
### 7.1 Influence du conditionnement
**Exercice 7.1.** Pour $f_\kappa(x, y) = \tfrac12 (x^2 + \kappa y^2)$ avec $\kappa \in \{1, 10, 100\}$, depuis $(1\,;\,1/\kappa)$, comparer le nombre d'itérations (critère `"grad_rel"`, `tol=1e-6`) du pas fixe $2/(1+\kappa)$ et du backtracking.

In [ ]:
print(" kappa | pas fixe | backtracking")
for k in [1, 10, 100]:
    # À COMPLÉTER
    pass

**Question 7.2.** Comment le nombre d'itérations évolue-t-il avec $\kappa$ ?

**Réponse :** *(à compléter)*

### 7.2 La fonction de Rosenbrock
$f(x, y) = (1 - x)^2 + 100 (y - x^2)^2$, de minimum $(1, 1)$, depuis $(-1{,}2\,;\,1)$.

**Exercice 7.3.** Comparer le pas fixe $10^{-3}$ (`max_iter=20000`) et le backtracking (`max_iter=50000`), critère `"grad"`, `tol=1e-6`. Afficher statut, itérations et point obtenu.

In [ ]:
rosen = lambda p: (1 - p[0])**2 + 100 * (p[1] - p[0]**2)**2
rgrad = lambda p: np.array([-2 * (1 - p[0]) - 400 * p[0] * (p[1] - p[0]**2), 200 * (p[1] - p[0]**2)])
r0 = np.array([-1.2, 1.0])
# À COMPLÉTER

**Question 7.4.** Que constatez-vous ? Pourquoi cette fonction est-elle difficile pour la descente de gradient ?

**Réponse :** *(à compléter)*

### 7.3 Application : régression linéaire
On dispose de $N = 50$ points et on cherche la droite $y = a x + b$ qui minimise $L(a, b) = \frac{1}{N} \sum_i (a x_i + b - y_i)^2$, de gradient $\left(\frac{2}{N}\sum (a x_i + b - y_i) x_i,\; \frac{2}{N}\sum (a x_i + b - y_i)\right)^\top$.

**Exercice 7.5.** Écrire `L(p)` et `grad_L(p)` pour $p = (a, b)$, lancer la descente (pas $0{,}1$, `max_iter=5000`, `tol=1e-8`, critère `"grad"`) depuis $(0, 0)$ et comparer avec `np.polyfit`. Essayer ensuite les pas $0{,}01$, $0{,}5$ et $0{,}9$.

In [ ]:
rng = np.random.default_rng(0)
xd = np.linspace(0, 1, 50)
yd = 2 * xd + 1 + 0.1 * rng.normal(size=50)

def L(p):
    # À COMPLÉTER
    raise NotImplementedError("À compléter")
def grad_L(p):
    # À COMPLÉTER
    raise NotImplementedError("À compléter")

# À COMPLÉTER : descente, comparaison avec np.polyfit(xd, yd, 1), puis essais avec d'autres pas

**Question 7.6.** Que représente le pas ici (*learning rate*) ? Quelle limite observe-t-on ?

**Réponse :** *(à compléter)*

## Partie 8 : synthèse
**Question 8.1.** Vous devez écrire une fonction de descente de gradient pour un collègue qui ne connaît pas la fonction à minimiser à l'avance. Quel critère d'arrêt lui recommandez-vous, avec quelles protections ? Justifiez avec ce que vous avez observé aux parties 3.2 à 3.9.

**Réponse :** *(à compléter)*

**Question 8.2.** Résumez en quelques lignes comment choisir entre pas fixe, backtracking et pas optimal.

**Réponse :** *(à compléter)*

**Question 8.3.** Quelle est, selon vous, l'erreur la plus dangereuse que vous avez rencontrée dans ce TP, et comment l'éviter ?

**Réponse :** *(à compléter)*